In [1]:
import pyspark
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework_module_6') \
    .getOrCreate()
print(spark.version)

4.1.1


In [2]:
!powershell -command "Invoke-WebRequest -Uri https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -OutFile yellow_tripdata_2025-11.parquet"

In [3]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [4]:
df.repartition(4).write.parquet('yellow_partitioned', mode='overwrite')

In [5]:
from pyspark.sql import functions as F

count_nov_15 = df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15') \
    .count()

print(f"Number of trips on November 15th: {count_nov_15:,}")

Number of trips on November 15th: 162,604


In [6]:
df_with_duration = df.withColumn('duration_hrs', 
    (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime)) / 3600
)
max_duration = df_with_duration.select(F.max('duration_hrs')).collect()[0][0]

print(f"The longest trip was {max_duration:.2f} hours")

The longest trip was 90.65 hours


In [7]:
import urllib.request
z_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
urllib.request.urlretrieve(z_url, "taxi_zone_lookup.csv")

# Now load it into Spark
df_zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

In [8]:
df.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

# Run the SQL query
spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(1) as trip_count
    FROM 
        trips t
    JOIN 
        zones z ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        trip_count ASC
    LIMIT 10
""").show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|Eltingville/Annad...|         1|
|       Arden Heights|         1|
|       Port Richmond|         3|
|       Rikers Island|         4|
|   Rossville/Woodrow|         4|
|         Great Kills|         4|
| Green-Wood Cemetery|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
+--------------------+----------+



In [9]:
# Join the datasets on the ID columns
# df is your yellow taxi data, df_zones is the lookup table
df_joined = df.join(df_zones, df.PULocationID == df_zones.LocationID)

# Group by the Zone name, count them, and sort from smallest to largest
least_frequent_zone = df_joined \
    .groupBy('Zone') \
    .count() \
    .orderBy('count', ascending=True)

# Show the top 5 to see the winner
least_frequent_zone.show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows
